<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/prepare_hst_yolo_10k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare the 10K HST dataset for YOLOv8-Pose

This notebook runs the preprocessing script from the GitHub repository. It reads the original DLVS3-HST PNG folders and metadata from Google Drive, then writes a **separate** YOLO dataset with an 80/10/10 split (seed 42).

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Get the updated preprocessing script

In [2]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/spacecraft-pose-estimation")
URL = "https://github.com/EliasNoorzad/spacecraft-pose-estimation.git"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

SCRIPT = REPO / "src" / "prepare_hst_yolo.py"
assert SCRIPT.is_file(), f"Preprocessing script not found: {SCRIPT}"
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "Pillow"], check=True)
print(f"Script ready: {SCRIPT}")

Script ready: /content/spacecraft-pose-estimation/src/prepare_hst_yolo.py


## 3. Check the input folders

The uploaded dataset is expected at `MyDrive/DLVS3-HST/`, with ten numbered folders (`0000000` through `0009000`) and their matching `*_metadata.json` files.

In [3]:
SOURCE = Path("/content/drive/MyDrive/DLVS3-HST")
OUTPUT = Path("/content/drive/MyDrive/hst_yolo_pose_10k")

assert SOURCE.is_dir(), f"Dataset folder not found: {SOURCE}"
groups = [f"{i:04d}000" for i in range(10)]  # 0000000, 0001000, ..., 0009000
missing = [name for name in groups if not (SOURCE / name).is_dir()]
missing += [f"{name}_metadata.json" for name in groups if not (SOURCE / f"{name}_metadata.json").is_file()]
assert not missing, "Missing dataset folders/files: " + ", ".join(missing)

print("Found all 10 image folders and 10 metadata JSON files.")
print("Source:", SOURCE)
print("Output:", OUTPUT)
if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise FileExistsError(f"Output folder is not empty: {OUTPUT}. Choose a new output path; existing data will not be overwritten.")

Found all 10 image folders and 10 metadata JSON files.
Source: /content/drive/MyDrive/DLVS3-HST
Output: /content/drive/MyDrive/hst_yolo_pose_10k


## 4. Convert all groups to YOLO pose format

The script matches annotations to images, creates labels for 37 keypoints, checks their coordinates, and globally splits the data 80/10/10. It stops if required images or annotations are missing or inconsistent. The output is written directly to Google Drive.

In [4]:
subprocess.run(
    [sys.executable, "-u", str(SCRIPT), "--root", str(SOURCE), "--output", str(OUTPUT)],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-u', '/content/spacecraft-pose-estimation/src/prepare_hst_yolo.py', '--root', '/content/drive/MyDrive/DLVS3-HST', '--output', '/content/drive/MyDrive/hst_yolo_pose_10k'], returncode=0)

## 5. Verify the output

In [5]:
expected = {"train": 8000, "val": 1000, "test": 1000}
for split, count in expected.items():
    images = sorted((OUTPUT / "images" / split).glob("*.png"))
    labels = sorted((OUTPUT / "labels" / split).glob("*.txt"))
    print(f"{split:5s}: {len(images):5d} images, {len(labels):5d} labels")
    assert len(images) == len(labels) == count, f"Unexpected count for {split}"
    assert {p.stem for p in images} == {p.stem for p in labels}, f"Image-label mismatch in {split}"

assert (OUTPUT / "data.yaml").is_file(), "data.yaml is missing"
print("\nAll 10,000 images have matching labels. The dataset is ready for YOLO training.")
print("YOLO config:", OUTPUT / "data.yaml")
if (OUTPUT / "filename_map.json").is_file():
    print("Renamed filename mapping:", OUTPUT / "filename_map.json")

train:  8000 images,  8000 labels
val  :  1000 images,  1000 labels
test :  1000 images,  1000 labels

All 10,000 images have matching labels. The dataset is ready for YOLO training.
YOLO config: /content/drive/MyDrive/hst_yolo_pose_10k/data.yaml


In [8]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
